# Quick Masked Training Smoke Test

Builds a small 3D U-Net, wraps it with a masked loss, and runs a single
training step on dummy data to verify everything works.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

from CosmoRecon.models import MaskedUNet3D, MaskedMSE

In [ ]:
# Dummy data
batch_size = 2
field_size = 16

x_dummy = np.random.rand(batch_size, field_size, field_size, field_size, 1).astype(np.float32)
y_dummy = np.random.rand(batch_size, field_size, field_size, field_size, 1).astype(np.float32)

# Binary mask: 1 = observed, 0 = missing (hole in the centre)
mask = np.ones((field_size, field_size, field_size), dtype=np.float32)
mask[4:12, 4:12, 4:12] = 0.0
mask_tf = tf.constant(mask[np.newaxis, ..., np.newaxis], dtype=tf.float32)

In [ ]:
# Build model
inpainter = MaskedUNet3D(
    input_size=(field_size, field_size, field_size, 1),
    base_filters=4,
    min_size=4,
    input_field="rho",
    norm_val=20.0,
    use_mask=True,
)
model = inpainter.unet

# Compile with masked loss
loss_fn = MaskedMSE(mask=mask_tf)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=loss_fn)
print(f"Model parameters: {model.count_params():,}")

In [ ]:
# Single training step
dataset = tf.data.Dataset.from_tensor_slices((x_dummy, y_dummy)).batch(batch_size)
history = model.fit(dataset, epochs=1, verbose=1)

print(f"Loss after 1 step: {history.history['loss'][0]:.6f}")
print("Test step ran successfully.")